In [1]:
import importlib
import sys

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

In [2]:
# Load the data
file_path_train = '../../../../../../data/Procurement/tensor_data/normal/procurement_all_5_train.pkl'
procurement_train_dataset = torch.load(file_path_train, weights_only=False)

file_path_val = '../../../../../../data/Procurement/tensor_data/normal/procurement_all_5_val.pkl'
procurement_val_dataset = torch.load(file_path_val, weights_only=False)

In [3]:
# Procurement dataset dynamic categories, features:
procurement_all_categories = procurement_train_dataset.all_categories

procurement_all_categories_cat = procurement_all_categories[0]
procurement_all_categories_num = procurement_all_categories[1]
for i, cat in enumerate(procurement_all_categories_cat):
     print(f"Procurement dynamic categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Procurement amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(procurement_all_categories_num):
     print(f"Procurement dynamic numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Procurement amount of numerical: {num[1]}")
print('\n')
     
# Procurement dataset static categories, features:
procurement_all_stat_categories = procurement_train_dataset.all_static_categories

procurement_all_stat_categories_cat = procurement_all_stat_categories[0]
procurement_all_stat_categories_num = procurement_all_stat_categories[1]
for i, cat in enumerate(procurement_all_stat_categories_cat):
     print(f"Procurement static categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Procurement amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(procurement_all_stat_categories_num):
     print(f"Procurement static numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Procurement amount of numerical: {num[1]}")

Procurement dynamic categorical feature: concept:name, Index position in categorical data list: 0
Procurement amount of category labels: 18
Procurement dynamic categorical feature: org:resource, Index position in categorical data list: 1
Procurement amount of category labels: 22
Procurement dynamic categorical feature: budget_status, Index position in categorical data list: 2
Procurement amount of category labels: 4
Procurement dynamic categorical feature: supplier_type, Index position in categorical data list: 3
Procurement amount of category labels: 6
Procurement dynamic categorical feature: goods_match, Index position in categorical data list: 4
Procurement amount of category labels: 5


Procurement dynamic numerical feature: case_elapsed_time, Index position in categorical data list: 0
Procurement amount of numerical: 1
Procurement dynamic numerical feature: event_elapsed_time, Index position in categorical data list: 1
Procurement amount of numerical: 1
Procurement dynamic numeric

In [4]:
# Create lists with name of Encoder features (input) and decoder features (input & output)
# Encoder features:
enc_feat_cat = []
enc_feat_num = []
for cat in procurement_all_categories_cat:
    enc_feat_cat.append(cat[0])
for num in procurement_all_categories_num:
    enc_feat_num.append(num[0])
enc_feat = [enc_feat_cat, enc_feat_num]
print("Input features encoder: ", enc_feat)

# Static encoder features:
stat_enc_feat_cat = []
stat_enc_feat_num = []
for cat in procurement_all_stat_categories_cat:
     stat_enc_feat_cat.append(cat[0])
for num in procurement_all_stat_categories_num:
     stat_enc_feat_num.append(num[0])
stat_enc_feat = [stat_enc_feat_cat, stat_enc_feat_num]
print("Input features static encoder: ", stat_enc_feat)

# Decoder features:
dec_feat_cat = ['concept:name']
dec_feat_num = []
dec_feat = [dec_feat_cat, dec_feat_num]
print("Features decoder: ", dec_feat)

Input features encoder:  [['concept:name', 'org:resource', 'budget_status', 'supplier_type', 'goods_match'], ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day', 'amount', 'revision_count', 'invoice_deviation_pct']]
Input features static encoder:  [['requester_seniority', 'department', 'category', 'priority'], []]
Features decoder:  [['concept:name'], []]


In [5]:
import suffix_pred.models.K_UED_LSTM
importlib.reload(suffix_pred.models.K_UED_LSTM)
from suffix_pred.models.K_UED_LSTM import DropoutUncertaintyEncoderDecoderLSTM

# Prediction decoder output sequence length
seq_len_pred = procurement_train_dataset.min_suffix_size

# Size hidden layer
hidden_size = 128

# Number of cells
num_layers = 4

# Fixed Dropout probability 
dropout = 0.1

# Encoder Decoder model initialization
model = DropoutUncertaintyEncoderDecoderLSTM(data_set_categories=procurement_all_categories,
                                             enc_feat=enc_feat,
                                             dec_feat=dec_feat,
                                             seq_len_pred=seq_len_pred,
                                             hidden_size=hidden_size,
                                             num_layers=num_layers,
                                             dropout=dropout,
                                             # static attributes
                                             static_data_set_categories=procurement_all_stat_categories,
                                             static_enc_feat=stat_enc_feat
                                             )

Dynamic data set categories:  ([('concept:name', 18, {'Approve Invoice': 1, 'Approve Requisition': 2, 'Close Case': 3, 'Collect Quotations': 4, 'Create Purchase Order': 5, 'Create Purchase Requisition': 6, 'EOS': 7, 'Evaluate Quotations': 8, 'Flag Invoice Mismatch': 9, 'Pay Invoice': 10, 'Receive Goods': 11, 'Reject Requisition': 12, 'Reorder Goods': 13, 'Request Credit Note': 14, 'Revise Requisition': 15, 'Select Supplier': 16, 'Send Purchase Order': 17}), ('org:resource', 22, {'Alice': 1, 'Bob': 2, 'Buyer_1': 3, 'Buyer_2': 4, 'Buyer_3': 5, 'Carol': 6, 'Clerk_1': 7, 'Clerk_2': 8, 'Clerk_3': 9, 'David': 10, 'EOS': 11, 'Eva': 12, 'Frank': 13, 'Manager_FIN_1': 14, 'Manager_FIN_2': 15, 'Manager_IT_1': 16, 'Manager_IT_2': 17, 'Manager_OPS_1': 18, 'Manager_OPS_2': 19, 'Receiver_A': 20, 'Receiver_B': 21}), ('budget_status', 4, {'EOS': 1, 'approved': 2, 'pending': 3}), ('supplier_type', 6, {'EOS': 1, 'preferred': 2, 'risky': 3, 'standard': 4, nan: 5}), ('goods_match', 5, {'EOS': 1, 'False': 2

In [6]:
import suffix_pred.loss
importlib.reload(suffix_pred.loss)
from suffix_pred.loss import Loss

loss_obj = Loss()

In [7]:
import suffix_pred.train
importlib.reload(suffix_pred.train)
from suffix_pred.train import UEDTrainer

from torch.optim.lr_scheduler import ReduceLROnPlateau

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Start learning rate
learning_rate = 1e-5
weight_decay = 0.0

# Optimizer and Scheduler
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=15, min_lr=1e-8)


# Keep consistent across all models
# Epochs / Batch size
num_epochs = 100
batch_size = 128

# L2 regularization term
regularization_term = 1e-4

# Shuffle data
shuffle = True

# Teacher forcing schedule
max_teacher_forcing_value = 1.0
min_teacher_forcing_value = 0.0

optimize_values = {"regularization_term": regularization_term,
                   "optimizer": optimizer,
                   "scheduler": scheduler,
                   "epochs": num_epochs,
                   "mini_batches": batch_size,
                   "shuffle": shuffle,
                   "min_teacher_forcing_value": min_teacher_forcing_value,
                   "max_teacher_forcing_value": max_teacher_forcing_value}

required_optimize_keys = {"regularization_term",
                          "optimizer",
                          "scheduler",
                          "epochs",
                          "mini_batches",
                          "shuffle",
                          "min_teacher_forcing_value",
                          "max_teacher_forcing_value"}

missing_keys = required_optimize_keys.difference(optimize_values.keys())
if missing_keys:
    raise ValueError(f"Missing required optimize_values keys: {sorted(missing_keys)}")

suffix_data_split_value = procurement_train_dataset.min_suffix_size
print("Train seq length:", suffix_data_split_value)

import os
model_output_path = "../../../../../../models/Procurement/clean/Procurement_UED_LSTM_v1_clean.pkl"
os.makedirs(os.path.dirname(model_output_path), exist_ok=True)

trainer = UEDTrainer(device=device,
                     model=model,
                     data_train=procurement_train_dataset,
                     data_val=procurement_val_dataset,
                     loss_obj=loss_obj,
                     log_normal_loss_num_feature=[],
                     optimize_values=optimize_values,
                     suffix_data_split_value=suffix_data_split_value,
                     save_model_n_th_epoch=1,
                     saving_path=model_output_path)

# Train the model
train_attenuated_losses, val_losses, val_attenuated_losses = trainer.train_model(use_statics=True,
                                                                                 use_zero_padd_masking=True,
                                                                                 use_eos_padd_masking=True)

Device: cuda
Train seq length: 5
Device:  cuda
Model:  DropoutUncertaintyEncoderDecoderLSTM(
  (embeddings_enc): ModuleList(
    (0): Embedding(18, 8)
    (1): Embedding(22, 9)
    (2): Embedding(4, 3)
    (3): Embedding(6, 4)
    (4): Embedding(5, 4)
  )
  (embeddings_static_enc): ModuleList(
    (0): Embedding(3, 3)
    (1): Embedding(4, 3)
    (2): Embedding(6, 4)
    (3): Embedding(5, 4)
  )
  (encoder): DropoutUncertaintyLSTMEncoder(
    (embeddings): ModuleList(
      (0): Embedding(18, 8)
      (1): Embedding(22, 9)
      (2): Embedding(4, 3)
      (3): Embedding(6, 4)
      (4): Embedding(5, 4)
    )
    (static_embeddings): ModuleList(
      (0): Embedding(3, 3)
      (1): Embedding(4, 3)
      (2): Embedding(6, 4)
      (3): Embedding(5, 4)
    )
    (input_proj): Linear(in_features=49, out_features=128, bias=True)
    (layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (act): ReLU()
    (first_layer): DropoutUncertaintyLSTMCell(
      (Wi): Linear(in_featu

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100], Learning Rate: 1e-05, Teacher forcing ratio: 1.0000, Scheduled sampling epsilon: 0.0000
Training: Avg Attenuated Training Loss: 3.4490
Validation: Avg Standard Validation Loss: 2.6552
Validation: Avg Attenuated Validation Loss: 2.8798
Validation Loss for Scheduler: 2.6552
saving model
Epoch [2/100], Learning Rate: 1e-05, Teacher forcing ratio: 0.9905, Scheduled sampling epsilon: 0.0095
Training: Avg Attenuated Training Loss: 2.1033
Validation: Avg Standard Validation Loss: 1.6206
Validation: Avg Attenuated Validation Loss: 1.7242
Validation Loss for Scheduler: 1.6206
saving model
Epoch [3/100], Learning Rate: 1e-05, Teacher forcing ratio: 0.9803, Scheduled sampling epsilon: 0.0197
Training: Avg Attenuated Training Loss: 1.0929
Validation: Avg Standard Validation Loss: 0.9567
Validation: Avg Attenuated Validation Loss: 1.0101
Validation Loss for Scheduler: 0.9567
saving model
Epoch [4/100], Learning Rate: 1e-05, Teacher forcing ratio: 0.9692, Scheduled sampling epsilon: 0